In [1]:
# MODIFIED SCRIPT: enhanced_polynomial_model_with_ckm_cubic_up.py
"""
Enhanced Polynomial Mass Generation Model with CKM Matrix and CUBIC Up-Quark Fit

This module implements an enhanced version of the polynomial mass generation approach
that includes all six quarks, CKM matrix calculations, and uses a CUBIC polynomial
for the up-type quarks (u, c, t) to avoid a separate top quark enhancement term.

Author: Manus AI / Modified by Assistant
Date: April 22, 2025
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import pandas as pd
# from scipy.linalg import expm # Not used in the original script provided

class EnhancedPolynomialModelCubicUp: # <<< CHANGE: Renamed class slightly
    """
    An enhanced implementation of the polynomial mass generation approach
    that includes all six quarks, CKM matrix calculations, and uses a CUBIC
    polynomial fit for up-type quarks.
    """

    def __init__(self):
        """
        Initialize the enhanced polynomial model with lattice QCD parameters.
        """
        # Lattice QCD parameters (from provided papers and PDG)
        self.alpha_s_mz = 0.11803 # Strong coupling at Z-boson mass

        # Quark masses at reference scales (GeV)
        self.mc_mc = 1.2735 # Charm quark mass at its own scale
        self.mb_mb = 4.188 # Bottom quark mass at its own scale
        # PDG values for other quarks (GeV)
        self.mu_2gev = 0.00216 # Up quark mass at 2 GeV
        self.md_2gev = 0.00467 # Down quark mass at 2 GeV
        self.ms_2gev = 0.093 # Strange quark mass at 2 GeV
        self.mt_mt = 172.76 # Top quark mass at its own scale

        # Reference scales (GeV)
        self.mu_ref_light = 2.0 # Reference scale for light quarks (u, d, s)
        self.mu_c = self.mc_mc # Reference scale for charm quark
        self.mu_b = self.mb_mb # Reference scale for bottom quark
        self.mu_t = self.mt_mt # Reference scale for top quark
        self.mz = 91.1876 # Z-boson mass

        # --- Polynomial coefficients (to be optimized) ---
        # <<< CHANGE: Up-type uses CUBIC (4 coeffs: a*L^3 + b*L^2 + c*L + d)
        self.poly_degree_up = 3
        self.c_coeffs_up_cubic = np.ones(self.poly_degree_up + 1) * 0.1 # Initial guess [a_u, b_u, c_u, d_u]

        # <<< CHANGE: Down-type remains QUADRATIC (3 coeffs: a*L^2 + b*L + c)
        self.poly_degree_down = 2
        self.c_coeffs_down = np.ones(self.poly_degree_down + 1) * 0.1 # Initial guess [a_d, b_d, c_d]

        # <<< REMOVED: Special top quark enhancement parameters
        # self.top_enhancement_factor = 10.0
        # self.top_exponent = 1.5

        # Initialize geodesic lengths (to be optimized)
        self.L_u = 0.1 # Initial guess for up quark geodesic length
        self.L_d = 0.2 # Initial guess for down quark geodesic length
        self.L_s = 0.5 # Initial guess for strange quark geodesic length
        self.L_c = 1.0 # Initial guess for charm quark geodesic length
        self.L_b = 2.0 # Initial guess for bottom quark geodesic length
        self.L_t = 3.0 # Initial guess for top quark geodesic length

        # Initialize geodesic angles for CKM matrix (to be optimized)
        self.theta_12 = 0.2 # Initial guess for Cabibbo angle
        self.theta_13 = 0.01 # Initial guess for theta_13
        self.theta_23 = 0.04 # Initial guess for theta_23
        self.delta_cp = 1.2 # Initial guess for CP-violating phase

        # Experimental CKM matrix magnitudes (PDG 2022)
        self.ckm_exp = np.array([
            [0.97435, 0.22500, 0.00369],
            [0.22486, 0.97349, 0.04182],
            [0.00857, 0.04110, 0.99915]
        ])

        # Beta function coefficients (unchanged)
        self.beta0_nf3 = (11.0 - 2.0/3.0 * 3.0) / 4.0
        self.beta1_nf3 = (102.0 - 38.0/3.0 * 3.0) / 16.0
        self.beta0_nf4 = (11.0 - 2.0/3.0 * 4.0) / 4.0
        self.beta1_nf4 = (102.0 - 38.0/3.0 * 4.0) / 16.0
        self.beta0_nf5 = (11.0 - 2.0/3.0 * 5.0) / 4.0
        self.beta1_nf5 = (102.0 - 38.0/3.0 * 5.0) / 16.0
        self.beta0_nf6 = (11.0 - 2.0/3.0 * 6.0) / 4.0
        self.beta1_nf6 = (102.0 - 38.0/3.0 * 6.0) / 16.0

        # Anomalous dimension coefficients (unchanged)
        self.gamma0 = 1.0
        self.gamma1_nf3 = (202.0/3.0 - 20.0/9.0 * 3.0) / 16.0
        self.gamma1_nf4 = (202.0/3.0 - 20.0/9.0 * 4.0) / 16.0
        self.gamma1_nf5 = (202.0/3.0 - 20.0/9.0 * 5.0) / 16.0
        self.gamma1_nf6 = (202.0/3.0 - 20.0/9.0 * 6.0) / 16.0

        # Generation scaling factors (unchanged)
        self.gen_scale = [1.0, 1.0, 1.0] # To be optimized

        # Optimization status
        self.is_optimized = False
        self.results = {}

        # Quark information (unchanged)
        self.quark_info = {
            'u': {'type': 'up', 'generation': 1, 'ref_mass': self.mu_2gev, 'ref_scale': self.mu_ref_light},
            'd': {'type': 'down', 'generation': 1, 'ref_mass': self.md_2gev, 'ref_scale': self.mu_ref_light},
            's': {'type': 'down', 'generation': 2, 'ref_mass': self.ms_2gev, 'ref_scale': self.mu_ref_light},
            'c': {'type': 'up', 'generation': 2, 'ref_mass': self.mc_mc, 'ref_scale': self.mu_c},
            'b': {'type': 'down', 'generation': 3, 'ref_mass': self.mb_mb, 'ref_scale': self.mu_b},
            't': {'type': 'up', 'generation': 3, 'ref_mass': self.mt_mt, 'ref_scale': self.mu_t}
        }

    # --- Methods for alpha_s, running_mass, calculate_ckm_matrix ---
    # --- remain unchanged from the original script             ---
    # --- (alpha_s, running_mass, calculate_ckm_matrix)         ---

    def alpha_s(self, mu):
        """Calculate the strong coupling constant at scale mu using 2-loop approximation."""
        # Determine number of active flavors
        if mu < 1.3: nf = 3; beta0 = self.beta0_nf3; beta1 = self.beta1_nf3
        elif mu < 4.2: nf = 4; beta0 = self.beta0_nf4; beta1 = self.beta1_nf4
        elif mu < 173.0: nf = 5; beta0 = self.beta0_nf5; beta1 = self.beta1_nf5
        else: nf = 6; beta0 = self.beta0_nf6; beta1 = self.beta1_nf6

        if abs(mu - self.mz) < 0.1: return self.alpha_s_mz

        if mu > 10.0:
            t = np.log(mu**2 / self.mz**2)
            # Simplified 1-loop for stability in this example region
            # Original had beta1 term, keep if needed: - beta1*np.log(abs(t) + 1.0)
            alpha_s_inv = 1.0/self.alpha_s_mz + beta0*t
            return 1.0 / max(alpha_s_inv, 1e-10)

        if mu < 10.0:
            mu_eff = max(mu, 1.0) # Freeze below 1 GeV
            if nf == 3: return max(0.1, 0.35 - 0.05 * np.log(mu_eff))
            elif nf == 4: return max(0.1, 0.25 - 0.02 * np.log(mu_eff))
            else: return max(0.1, 0.20 - 0.01 * np.log(mu_eff))
        return self.alpha_s_mz # Fallback

    def running_mass(self, m_ref, mu_ref, mu, nf):
        """Calculate the running mass at scale mu."""
        # Select appropriate coefficients
        if nf == 3: gamma0 = self.gamma0; gamma1 = self.gamma1_nf3; beta0 = self.beta0_nf3
        elif nf == 4: gamma0 = self.gamma0; gamma1 = self.gamma1_nf4; beta0 = self.beta0_nf4
        elif nf == 5: gamma0 = self.gamma0; gamma1 = self.gamma1_nf5; beta0 = self.beta0_nf5
        else: gamma0 = self.gamma0; gamma1 = self.gamma1_nf6; beta0 = self.beta0_nf6

        if abs(mu - mu_ref) < 0.01: return m_ref

        alpha_ref = self.alpha_s(mu_ref)
        alpha_mu = self.alpha_s(mu)
        alpha_ref = max(alpha_ref, 0.01)
        alpha_mu = max(alpha_mu, 0.01)

        # Simplified power-law approximation (as in original)
        power = gamma0 / (2 * beta0) if beta0 != 0 else 0
        ratio = alpha_mu / alpha_ref

        try:
            m_mu = m_ref * ratio**power
            # Apply bounds (simplified from original for brevity)
            if mu > mu_ref: m_mu = min(m_mu, m_ref * 1.1) # Mass decreases
            else: m_mu = max(m_mu, m_ref * 0.9) # Mass increases
            return max(m_mu, 1e-6) # Ensure positive
        except (ValueError, ZeroDivisionError, OverflowError):
            # Fallback (simplified linear)
             return m_ref * (1.0 - 0.1 * np.log10(max(1, mu / mu_ref))) if mu > mu_ref else m_ref * (1.0 + 0.1 * np.log10(max(1, mu_ref / mu)))


    def calculate_ckm_matrix(self):
        """Calculate the CKM matrix using the standard parameterization."""
        s12, c12 = np.sin(self.theta_12), np.cos(self.theta_12)
        s13, c13 = np.sin(self.theta_13), np.cos(self.theta_13)
        s23, c23 = np.sin(self.theta_23), np.cos(self.theta_23)
        delta = self.delta_cp
        exp_neg_id = np.exp(-1j * delta)
        exp_id = np.exp(1j * delta)

        ckm = np.array([
            [c12 * c13, s12 * c13, s13 * exp_neg_id],
            [-s12 * c23 - c12 * s23 * s13 * exp_id, c12 * c23 - s12 * s23 * s13 * exp_id, s23 * c13],
            [s12 * s23 - c12 * c23 * s13 * exp_id, -c12 * s23 - s12 * c23 * s13 * exp_id, c23 * c13]
        ])
        return ckm

    # <<< REMOVED: calculate_ckm_from_geodesics (was unused in optimization objective)

    def calculate_mass(self, quark, L=None):
        """
        Calculate quark mass from geodesic length using polynomial approach.
        Uses CUBIC for up-type, QUADRATIC for down-type.
        """
        quark_type = self.quark_info[quark]['type']
        generation = self.quark_info[quark]['generation']

        if L is None:
            L = getattr(self, f'L_{quark}')

        if quark_type == 'up':
            # <<< CHANGE: Use cubic coefficients [a, b, c, d]
            coeffs = self.c_coeffs_up_cubic
            # Calculate base mass using CUBIC polynomial: a*L^3 + b*L^2 + c*L + d
            mass = coeffs[0] * L**3 + coeffs[1] * L**2 + coeffs[2] * L + coeffs[3]
            # <<< REMOVED: Special enhancement for top quark was here
            # No separate top enhancement needed

        else: # quark_type == 'down'
            # <<< CHANGE: Use quadratic coefficients [a, b, c]
            coeffs = self.c_coeffs_down
            # Calculate base mass using QUADRATIC polynomial: a*L^2 + b*L + c
            mass = coeffs[0] * L**2 + coeffs[1] * L + coeffs[2]

        # Apply generation-specific scaling (same for both types)
        mass *= self.gen_scale[generation - 1]

        return max(mass, 1e-9) # Ensure mass is positive

    def optimize_parameters(self):
        """
        Optimize model parameters to match lattice QCD, PDG values, and CKM matrix.
        Uses CUBIC coefficients for up-type quarks.
        """
        def objective(params):
            # <<< CHANGE: Parameter extraction updated for cubic up-coeffs
            # Number of parameters: 4(up) + 3(down) + 6(L) + 3(gen) + 4(CKM) = 20
            self.c_coeffs_up_cubic = params[0:4]    # Indices 0, 1, 2, 3
            self.c_coeffs_down = params[4:7]        # Indices 4, 5, 6
            # <<< REMOVED: top_enhancement_factor, top_exponent parameters
            self.L_u = params[7]                    # Index 7
            self.L_d = params[8]                    # Index 8
            self.L_s = params[9]                    # Index 9
            self.L_c = params[10]                   # Index 10
            self.L_b = params[11]                   # Index 11
            self.L_t = params[12]                   # Index 12
            self.gen_scale = params[13:16]          # Indices 13, 14, 15
            self.theta_12 = params[16]              # Index 16
            self.theta_13 = params[17]              # Index 17
            self.theta_23 = params[18]              # Index 18
            self.delta_cp = params[19]              # Index 19

            # Calculate masses and errors (using the updated calculate_mass)
            masses = {}
            errors = {}
            for quark in self.quark_info:
                ref_mass = self.quark_info[quark]['ref_mass']
                try:
                    # Need to handle potential numerical issues during optimization
                    masses[quark] = self.calculate_mass(quark)
                    if ref_mass > 1e-9: # Avoid division by zero for reference mass
                         errors[quark] = ((masses[quark] - ref_mass) / ref_mass)**2
                    else:
                         errors[quark] = masses[quark]**2 # Penalize non-zero mass if ref is zero
                except (ValueError, OverflowError):
                    # Assign a large penalty if calculation fails
                    errors[quark] = 1e10
                    masses[quark] = np.inf


            # Calculate CKM matrix and errors
            ckm = self.calculate_ckm_matrix()
            ckm_mag = np.abs(ckm)
            # Ensure CKM magnitudes are valid numbers
            if np.any(np.isnan(ckm_mag)) or np.any(np.isinf(ckm_mag)):
                 ckm_errors = 1e10 # Large penalty
            else:
                 # Avoid division by zero in CKM errors for small elements
                 ckm_exp_safe = np.maximum(self.ckm_exp, 1e-6)
                 ckm_errors = np.sum(((ckm_mag - self.ckm_exp) / ckm_exp_safe)**2)


            # Total error with weighting (same weighting as original)
            total_error = (
                errors.get('u', 1e10) + errors.get('d', 1e10) + errors.get('s', 1e10) +
                5 * errors.get('c', 1e10) + 5 * errors.get('b', 1e10) + 20 * errors.get('t', 1e10) +
                10 * ckm_errors
            )

            # <<< CHANGE: Regularization updated for cubic coeffs
            # Regularization to prevent extreme parameter values
            regularization = 0.01 * (
                np.sum(self.c_coeffs_up_cubic**2) + # Use cubic coeffs
                np.sum(self.c_coeffs_down**2) +
                # Removed top enhancement terms from regularization
                np.sum((np.array([self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t]) -
                        np.array([0.1, 0.2, 0.5, 1.0, 2.0, 3.0]))**2) + # L deviation
                 np.sum((np.array(self.gen_scale) - 1.0)**2) + # Gen scale deviation from 1
                 (self.theta_12 - 0.2)**2 + (self.theta_13 - 0.01)**2 +
                 (self.theta_23 - 0.04)**2 + (self.delta_cp - 1.2)**2 # CKM angles deviation
            )

            # Ensure total error is a finite number
            if np.isnan(total_error) or np.isinf(total_error):
                return 1e20 # Return a large finite number if calculation failed

            return total_error + regularization

        # <<< CHANGE: Initial guess updated (4 up cubic coeffs, remove 2 top params)
        initial_guess = np.concatenate([
            self.c_coeffs_up_cubic,     # 4 parameters
            self.c_coeffs_down,         # 3 parameters
            # Removed top_enhancement_factor, top_exponent
            [self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t], # 6 parameters
            self.gen_scale,             # 3 parameters
            [self.theta_12, self.theta_13, self.theta_23, self.delta_cp] # 4 parameters
        ]) # Total 20 parameters

        # <<< CHANGE: Bounds updated
        bounds = []
        # Bounds for up-type CUBIC coefficients [a_u, b_u, c_u, d_u]
        bounds.extend([(-100.0, 100.0)] * (self.poly_degree_up + 1)) # Wider range for cubic
        # Bounds for down-type QUADRATIC coefficients [a_d, b_d, c_d]
        bounds.extend([(-100.0, 100.0)] * (self.poly_degree_down + 1)) # Wider range
        # <<< REMOVED: Bounds for top enhancement parameters

        # Geodesic lengths (same as original, consider widening if needed)
        bounds.append((0.01, 0.5)) # L_u
        bounds.append((0.01, 0.5)) # L_d
        bounds.append((0.1, 1.0)) # L_s
        bounds.append((0.5, 2.0)) # L_c
        bounds.append((1.0, 5.0)) # L_b (Increased upper bound slightly)
        bounds.append((2.0, 8.0)) # L_t (Increased upper bound slightly)

        # Generation scaling factors (same as original, consider widening)
        bounds.append((0.001, 10.0)) # gen_scale[0]
        bounds.append((0.01, 50.0))  # gen_scale[1] (Wider range)
        bounds.append((0.1, 500.0))  # gen_scale[2] (Wider range, might be large for top)

        # CKM parameters (same as original)
        bounds.append((0.1, 0.3)) # theta_12
        bounds.append((0.001, 0.05)) # theta_13
        bounds.append((0.01, 0.1)) # theta_23
        bounds.append((0.0, 2*np.pi)) # delta_cp

        # Ensure we have the correct number of bounds (should be 20)
        if len(bounds) != len(initial_guess):
             raise ValueError(f"Mismatch between number of parameters ({len(initial_guess)}) and bounds ({len(bounds)})")


        # Perform optimization with increased limits
        # <<< CHANGE: Added options for maxiter and maxfun
        result = minimize(objective, initial_guess, method='L-BFGS-B', bounds=bounds,
                          options={'maxiter': 50000, 'maxfun': 50000, 'ftol': 1e-9, 'gtol': 1e-6})

        # <<< CHANGE: Update parameters with optimized values using new indices
        if result.success:
            self.c_coeffs_up_cubic = result.x[0:4]
            self.c_coeffs_down = result.x[4:7]
            # Removed top factor/exponent update
            self.L_u = result.x[7]
            self.L_d = result.x[8]
            self.L_s = result.x[9]
            self.L_c = result.x[10]
            self.L_b = result.x[11]
            self.L_t = result.x[12]
            self.gen_scale = result.x[13:16]
            self.theta_12 = result.x[16]
            self.theta_13 = result.x[17]
            self.theta_23 = result.x[18]
            self.delta_cp = result.x[19]
            self.is_optimized = True
        else:
            print("Optimization did not converge. Using initial parameters for results.")
            # Keep initial parameters if optimization failed


        # Calculate final masses and errors with optimized (or initial) params
        masses = {}
        errors = {}
        for quark in self.quark_info:
            ref_mass = self.quark_info[quark]['ref_mass']
            pred_mass = self.calculate_mass(quark)
            masses[quark] = pred_mass
            if ref_mass > 1e-9:
                 errors[quark] = abs((pred_mass - ref_mass) / ref_mass) * 100 # Percentage
            else:
                 errors[quark] = abs(pred_mass) * 100 # Error relative to zero

        # Calculate final CKM matrix and errors
        ckm = self.calculate_ckm_matrix()
        ckm_mag = np.abs(ckm)
        ckm_exp_safe = np.maximum(self.ckm_exp, 1e-6)
        ckm_errors = np.abs((ckm_mag - self.ckm_exp) / ckm_exp_safe) * 100 # Percentage

        # Store results
        self.results = {
            'c_coeffs_up_cubic': self.c_coeffs_up_cubic, # <<< CHANGE: Storing cubic coeffs
            'c_coeffs_down': self.c_coeffs_down,
            # <<< REMOVED: top_enhancement_factor, top_exponent from results
            'L_u': self.L_u, 'L_d': self.L_d, 'L_s': self.L_s,
            'L_c': self.L_c, 'L_b': self.L_b, 'L_t': self.L_t,
            'gen_scale': self.gen_scale,
            'theta_12': self.theta_12, 'theta_13': self.theta_13,
            'theta_23': self.theta_23, 'delta_cp': self.delta_cp,
            'masses': masses,
            'errors': errors,
            'ckm': ckm_mag,
            'ckm_exp': self.ckm_exp,
            'ckm_errors': ckm_errors,
            'success': result.success,
            'message': result.message,
            'final_objective_value': result.fun if hasattr(result, 'fun') else objective(result.x if result.success else initial_guess)
        }

        # If optimization failed, make sure results reflect this
        if not result.success:
             self.results['success'] = False
             self.results['message'] = result.message


        return self.results

    # --- Methods for calculate_running_masses, calculate_alpha_s_values ---
    # --- remain unchanged from the original script                      ---
    def calculate_running_masses(self, mu_values):
        """Calculate running masses at different energy scales."""
        if not self.is_optimized:
            print("Warning: Parameters not optimized. Running optimization first.")
            self.optimize_parameters()
            if not self.is_optimized: # Check if optimization succeeded
                 print("Error: Optimization failed. Cannot calculate running masses.")
                 return {'mu': mu_values, **{q: [np.nan]*len(mu_values) for q in self.quark_info}}


        running_masses = {'mu': mu_values}
        for quark in self.quark_info:
            running_masses[quark] = []
            # Use the mass calculated by the model at its reference scale as m_ref
            ref_mass_pred = self.results['masses'][quark]
            ref_scale = self.quark_info[quark]['ref_scale']
            for mu in mu_values:
                # Determine number of active flavors (same logic as alpha_s)
                if mu < 1.3: nf = 3
                elif mu < 4.2: nf = 4
                elif mu < 173.0: nf = 5
                else: nf = 6
                m_mu = self.running_mass(ref_mass_pred, ref_scale, mu, nf)
                running_masses[quark].append(m_mu)
        return running_masses

    def calculate_alpha_s_values(self, mu_values):
        """Calculate strong coupling constant at different energy scales."""
        alpha_s_values = {'mu': mu_values, 'alpha_s': []}
        for mu in mu_values:
            alpha_s_values['alpha_s'].append(self.alpha_s(mu))
        return alpha_s_values

    # --- Methods for generate_report, plotting ---
    # --- Need updates for cubic coefficients and removed top enhancement ---

    def generate_report(self):
        """Generate a report of the model results (updated for cubic)."""
        if not self.is_optimized:
             print("Warning: Parameters not optimized. Running optimization first.")
             self.optimize_parameters()
             # No need to check success here, report will show optimization status

        report_path = "enhanced_model_cubic_up_report.md" # <<< CHANGE: New filename
        with open(report_path, 'w') as f:
            f.write("# Enhanced Polynomial Model with CKM Matrix (Cubic Up-Quark Fit) Report\n\n")
            f.write("## Optimization Status\n\n") # <<< Moved status to top
            f.write(f"Success: {self.results.get('success', 'N/A')}\n")
            f.write(f"Message: {self.results.get('message', 'N/A')}\n")
            f.write(f"Final Objective Function Value: {self.results.get('final_objective_value', 'N/A'):.6e}\n\n")

            f.write("## Optimized Parameters\n\n")
            # <<< CHANGE: Report cubic coefficients
            f.write("### Polynomial Coefficients for Up-type Quarks (Cubic: aL^3 + bL^2 + cL + d)\n\n")
            labels = ['a_u', 'b_u', 'c_u', 'd_u']
            for i, c in enumerate(self.results.get('c_coeffs_up_cubic', [np.nan]*4)):
                f.write(f"{labels[i]} = {c:.6f} \n") # Removed GeV units, inconsistent application previously

            # <<< CHANGE: Report quadratic coefficients for down-type
            f.write("\n### Polynomial Coefficients for Down-type Quarks (Quadratic: aL^2 + bL + c)\n\n")
            labels_down = ['a_d', 'b_d', 'c_d']
            for i, c in enumerate(self.results.get('c_coeffs_down', [np.nan]*3)):
                f.write(f"{labels_down[i]} = {c:.6f} \n")

            # <<< REMOVED: Top Quark Enhancement Parameters section

            f.write("\n### Generation Scaling Factors\n\n")
            for i, s in enumerate(self.results.get('gen_scale', [np.nan]*3)):
                f.write(f"Generation {i+1}: {s:.6f}\n")

            f.write("\n### Geodesic Lengths\n\n")
            for q in ['u', 'd', 's', 'c', 'b', 't']:
                 f.write(f"L_{q} = {self.results.get(f'L_{q}', np.nan):.6f}\n")

            f.write("\n### CKM Matrix Parameters\n\n")
            theta_12 = self.results.get('theta_12', np.nan)
            theta_13 = self.results.get('theta_13', np.nan)
            theta_23 = self.results.get('theta_23', np.nan)
            delta_cp = self.results.get('delta_cp', np.nan)
            f.write(f"θ₁₂ = {theta_12:.6f} rad = {np.degrees(theta_12):.4f}°\n")
            f.write(f"θ₁₃ = {theta_13:.6f} rad = {np.degrees(theta_13):.4f}°\n")
            f.write(f"θ₂₃ = {theta_23:.6f} rad = {np.degrees(theta_23):.4f}°\n")
            f.write(f"δ_CP = {delta_cp:.6f} rad = {np.degrees(delta_cp):.4f}°\n")

            f.write("\n## Mass Predictions at Reference Scales\n\n")
            f.write("| Quark | Reference Scale (GeV) | Predicted Mass (GeV) | Reference Value (GeV) | Error (%) |\n")
            f.write("|-------|----------------------|----------------------|----------------------|----------|\n")
            masses = self.results.get('masses', {})
            errors = self.results.get('errors', {})
            for quark in self.quark_info:
                ref_scale = self.quark_info[quark]['ref_scale']
                ref_mass = self.quark_info[quark]['ref_mass']
                pred_mass = masses.get(quark, np.nan)
                error = errors.get(quark, np.nan)
                f.write(f"| {quark:<5} | {ref_scale:20.4f} | {pred_mass:20.6f} | {ref_mass:20.6f} | {error:8.4f} |\n")


            f.write("\n## CKM Matrix\n\n")
            f.write("### Predicted CKM Matrix (Magnitudes)\n\n")
            f.write("```
            ckm_pred = self.results.get('ckm', np.full((3,3), np.nan))
            for i in range(3):
                f.write("[ ")
                for j in range(3): f.write(f"{ckm_pred[i, j]:.6f} ")
                f.write("]\n")
            f.write("```\n\n")

            f.write("### Experimental CKM Matrix (Magnitudes)\n\n")
            f.write("```
            for i in range(3):
                f.write("[ ")
                for j in range(3): f.write(f"{self.ckm_exp[i, j]:.6f} ")
                f.write("]\n")
            f.write("```\n\n")

            f.write("### CKM Matrix Errors (%)\n\n")
            f.write("```
            ckm_err = self.results.get('ckm_errors', np.full((3,3), np.nan))
            for i in range(3):
                f.write("[ ")
                for j in range(3): f.write(f"{ckm_err[i, j]:.4f} ")
                f.write("]\n")
            f.write("```\n\n")

            # --- Running Masses and Alpha_s sections (can remain similar) ---
            f.write("\n## Running Masses\n\n")
            # Add check if optimization succeeded before calculating/writing this
            if self.results.get('success', False):
                 mu_values_report = [1.0, 2.0, 5.0, 10.0, self.mz, self.mt_mt]
                 running_masses = self.calculate_running_masses(mu_values_report)
                 f.write("| μ (GeV) | m_u (GeV) | m_d (GeV) | m_s (GeV) | m_c (GeV) | m_b (GeV) | m_t (GeV) |\n")
                 f.write("|---------|-----------|-----------|-----------|-----------|-----------|----------|\n")
                 for i, mu in enumerate(running_masses['mu']):
                      f.write(f"| {mu:7.4f} | {running_masses['u'][i]:9.6f} | {running_masses['d'][i]:9.6f} | {running_masses['s'][i]:9.6f} | {running_masses['c'][i]:9.6f} | {running_masses['b'][i]:9.6f} | {running_masses['t'][i]:9.6f} |\n")
            else:
                 f.write("Running masses not calculated due to optimization failure.\n")


            f.write("\n## Strong Coupling Constant\n\n")
            if self.results.get('success', False):
                 alpha_s_values = self.calculate_alpha_s_values(mu_values_report)
                 f.write("| μ (GeV) | α_s |\n")
                 f.write("|---------|------|\n")
                 for i, mu in enumerate(alpha_s_values['mu']):
                      f.write(f"| {mu:7.4f} | {alpha_s_values['alpha_s'][i]:.6f} |\n")
            else:
                 f.write("Alpha_s values not calculated due to optimization failure.\n")

        return report_path

    # --- Plotting functions need significant updates ---
    # --- plot_running_masses, plot_alpha_s (minor changes maybe) ---
    # --- plot_polynomial_functions (major changes) ---
    # --- plot_geodesic_lengths, plot_mass_hierarchy (likely unchanged) ---
    # --- plot_ckm_matrix (likely unchanged) ---
    # --- create_comprehensive_visualization (major changes) ---

    def plot_polynomial_functions(self):
        """Plot the polynomial mass generation functions (updated for cubic)."""
        if not self.is_optimized:
             print("Warning: Run optimization first for meaningful plot."); return None, None
        if not self.results.get('success', False):
             print("Warning: Optimization failed. Plot may use initial parameters.")


        L_values = np.linspace(0, self.results.get('L_t', 5.0) * 1.1, 200) # Extend range slightly past L_t

        # <<< CHANGE: Calculate up-type masses using CUBIC
        coeffs_up = self.results.get('c_coeffs_up_cubic', [0,0,0,0])
        up_masses = coeffs_up[0] * L_values**3 + coeffs_up[1] * L_values**2 + coeffs_up[2] * L_values + coeffs_up[3]
        up_masses = np.maximum(up_masses, 1e-9) # Ensure positive for log plot

        # <<< CHANGE: Calculate down-type masses using QUADRATIC
        coeffs_down = self.results.get('c_coeffs_down', [0,0,0])
        down_masses = coeffs_down[0] * L_values**2 + coeffs_down[1] * L_values + coeffs_down[2]
        down_masses = np.maximum(down_masses, 1e-9) # Ensure positive

        # <<< REMOVED: Calculation of top enhancement curve

        # Get predicted masses at optimized L values (apply scaling factors)
        pred_masses = {}
        gen_scales = self.results.get('gen_scale', [1,1,1])
        Ls = {q: self.results.get(f'L_{q}', 0) for q in self.quark_info}

        try: # Recalculate with scaling for plotting points correctly
             pred_masses['u'] = (coeffs_up[0]*Ls['u']**3 + coeffs_up[1]*Ls['u']**2 + coeffs_up[2]*Ls['u'] + coeffs_up[3]) * gen_scales[0]
             pred_masses['c'] = (coeffs_up[0]*Ls['c']**3 + coeffs_up[1]*Ls['c']**2 + coeffs_up[2]*Ls['c'] + coeffs_up[3]) * gen_scales[1]
             pred_masses['t'] = (coeffs_up[0]*Ls['t']**3 + coeffs_up[1]*Ls['t']**2 + coeffs_up[2]*Ls['t'] + coeffs_up[3]) * gen_scales[2]
             pred_masses['d'] = (coeffs_down[0]*Ls['d']**2 + coeffs_down[1]*Ls['d'] + coeffs_down[2]) * gen_scales[0]
             pred_masses['s'] = (coeffs_down[0]*Ls['s']**2 + coeffs_down[1]*Ls['s'] + coeffs_down[2]) * gen_scales[1]
             pred_masses['b'] = (coeffs_down[0]*Ls['b']**2 + coeffs_down[1]*Ls['b'] + coeffs_down[2]) * gen_scales[2]
        except Exception as e:
             print(f"Error calculating predicted masses for plot points: {e}")
             # Fill with NaNs if calculation fails
             pred_masses = {q: np.nan for q in self.quark_info}


        # Create plot (Linear Scale)
        plt.figure(figsize=(10, 6))
        plt.plot(L_values, up_masses * gen_scales[0], 'r--', alpha=0.5, label='Up-type Poly * g1') # Show scaled polynomials
        plt.plot(L_values, up_masses * gen_scales[1], 'r-.', alpha=0.5, label='Up-type Poly * g2')
        plt.plot(L_values, up_masses * gen_scales[2], 'r-', alpha=0.8, label='Up-type Poly * g3 (Cubic)')
        plt.plot(L_values, down_masses * gen_scales[0], 'b--', alpha=0.5, label='Down-type Poly * g1')
        plt.plot(L_values, down_masses * gen_scales[1], 'b-.', alpha=0.5, label='Down-type Poly * g2')
        plt.plot(L_values, down_masses * gen_scales[2], 'b-', alpha=0.8, label='Down-type Poly * g3 (Quadratic)')
        # <<< REMOVED: Plot of combined up + top enhancement

        # Add quark points (using the recalculated masses with scaling)
        plt.scatter([Ls['u']], [pred_masses.get('u',np.nan)], color='red', marker='o', s=50, label='Up (pred)')
        plt.scatter([Ls['c']], [pred_masses.get('c',np.nan)], color='red', marker='s', s=50, label='Charm (pred)')
        plt.scatter([Ls['t']], [pred_masses.get('t',np.nan)], color='red', marker='^', s=50, label='Top (pred)') # Changed color
        plt.scatter([Ls['d']], [pred_masses.get('d',np.nan)], color='blue', marker='o', s=50, label='Down (pred)')
        plt.scatter([Ls['s']], [pred_masses.get('s',np.nan)], color='blue', marker='s', s=50, label='Strange (pred)')
        plt.scatter([Ls['b']], [pred_masses.get('b',np.nan)], color='blue', marker='^', s=50, label='Bottom (pred)')

        plt.xlabel('Geodesic Length L')
        plt.ylabel('Predicted Mass (GeV)')
        plt.title('Polynomial Mass Generation Functions (Scaled)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(fontsize='small')
        plt.ylim(bottom=-abs(pred_masses.get('t', 180))*0.1) # Adjust y-lim if needed

        plot_path = "enhanced_poly_cubic_functions_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        # Create plot (Log Scale)
        plt.figure(figsize=(10, 6))
        plt.semilogy(L_values, up_masses * gen_scales[0], 'r--', alpha=0.5, label='Up-type Poly * g1')
        plt.semilogy(L_values, up_masses * gen_scales[1], 'r-.', alpha=0.5, label='Up-type Poly * g2')
        plt.semilogy(L_values, up_masses * gen_scales[2], 'r-', alpha=0.8, label='Up-type Poly * g3 (Cubic)')
        plt.semilogy(L_values, down_masses * gen_scales[0], 'b--', alpha=0.5, label='Down-type Poly * g1')
        plt.semilogy(L_values, down_masses * gen_scales[1], 'b-.', alpha=0.5, label='Down-type Poly * g2')
        plt.semilogy(L_values, down_masses * gen_scales[2], 'b-', alpha=0.8, label='Down-type Poly * g3 (Quad)')

        plt.scatter([Ls['u']], [pred_masses.get('u',np.nan)], color='red', marker='o', s=50, label='Up (pred)')
        plt.scatter([Ls['c']], [pred_masses.get('c',np.nan)], color='red', marker='s', s=50, label='Charm (pred)')
        plt.scatter([Ls['t']], [pred_masses.get('t',np.nan)], color='red', marker='^', s=50, label='Top (pred)')
        plt.scatter([Ls['d']], [pred_masses.get('d',np.nan)], color='blue', marker='o', s=50, label='Down (pred)')
        plt.scatter([Ls['s']], [pred_masses.get('s',np.nan)], color='blue', marker='s', s=50, label='Strange (pred)')
        plt.scatter([Ls['b']], [pred_masses.get('b',np.nan)], color='blue', marker='^', s=50, label='Bottom (pred)')

        plt.xlabel('Geodesic Length L')
        plt.ylabel('Predicted Mass (GeV) - Log Scale')
        plt.title('Polynomial Mass Generation Functions (Scaled, Log Scale)')
        plt.grid(True, which='both', linestyle='--', alpha=0.7)
        plt.legend(fontsize='small')
        # Adjust y-lim for log scale, ensure min is positive
        min_mass = min(m for m in pred_masses.values() if m is not None and m > 1e-9)
        max_mass = max(m for m in pred_masses.values() if m is not None)
        if min_mass is not None and max_mass is not None:
             plt.ylim(bottom=min_mass * 0.1, top=max_mass * 10)

        log_plot_path = "enhanced_poly_cubic_functions_log_plot.png"
        plt.savefig(log_plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        return plot_path, log_plot_path

    # --- Other plotting functions (geodesic_lengths, mass_hierarchy, ckm_matrix) ---
    # --- are likely okay, but comprehensive plot needs update.                 ---

    def plot_geodesic_lengths(self):
        """Plot the geodesic lengths for all quarks."""
        if not self.is_optimized: print("Warning: Run optimization first."); return None
        if not self.results.get('success', False): print("Warning: Optimization failed.")

        plt.figure(figsize=(10, 6))
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        lengths = [self.results.get(f'L_{q}', np.nan) for q in quarks]
        colors = ['r', 'b', 'g', 'c', 'm', 'y']
        plt.bar(quarks, lengths, color=colors)
        plt.xlabel('Quark')
        plt.ylabel('Optimized Geodesic Length')
        plt.title('Geodesic Lengths for All Quarks')
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        plot_path = "enhanced_cubic_geodesic_lengths_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        return plot_path

    def plot_mass_hierarchy(self):
        """Plot the mass hierarchy for all quarks."""
        if not self.is_optimized: print("Warning: Run optimization first."); return None
        if not self.results.get('success', False): print("Warning: Optimization failed.")

        plt.figure(figsize=(10, 6))
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        masses = [self.results.get('masses', {}).get(q, np.nan) for q in quarks]
        # Filter out NaNs before plotting
        valid_indices = [i for i, m in enumerate(masses) if not np.isnan(m)]
        quarks_valid = [quarks[i] for i in valid_indices]
        masses_valid = [masses[i] for i in valid_indices]
        colors = ['r', 'b', 'g', 'c', 'm', 'y']
        colors_valid = [colors[i] for i in valid_indices]

        if masses_valid: # Only plot if there are valid masses
             plt.bar(quarks_valid, masses_valid, color=colors_valid)
             plt.yscale('log')
             plt.xlabel('Quark')
             plt.ylabel('Predicted Mass (GeV) - Log Scale')
             plt.title('Quark Mass Hierarchy (Predicted)')
             plt.grid(True, axis='y', which='both', linestyle='--', alpha=0.7)
        else:
             plt.title("Quark Mass Hierarchy (Predicted) - No valid data to plot")

        plot_path = "enhanced_cubic_mass_hierarchy_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        return plot_path

    def plot_ckm_matrix(self):
        """Plot the CKM matrix."""
        if not self.is_optimized: print("Warning: Run optimization first."); return None
        if not self.results.get('success', False): print("Warning: Optimization failed.")

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        ckm_pred = self.results.get('ckm', np.full((3,3), np.nan))
        ckm_err = self.results.get('ckm_errors', np.full((3,3), np.nan))

        # Plot 1: Predicted CKM matrix
        im1 = axes[0].imshow(ckm_pred, cmap='viridis', vmin=0, vmax=1)
        axes[0].set_title('Predicted CKM Matrix')
        for i in range(3):
            for j in range(3): axes[0].text(j, i, f"{ckm_pred[i, j]:.4f}", ha="center", va="center", color="w" if ckm_pred[i, j] < 0.5 else "k")

        # Plot 2: Experimental CKM matrix
        im2 = axes[1].imshow(self.ckm_exp, cmap='viridis', vmin=0, vmax=1)
        axes[1].set_title('Experimental CKM Matrix')
        for i in range(3):
            for j in range(3): axes[1].text(j, i, f"{self.ckm_exp[i, j]:.4f}", ha="center", va="center", color="w" if self.ckm_exp[i, j] < 0.5 else "k")

        # Plot 3: Error percentage
        im3 = axes[2].imshow(ckm_err, cmap='hot', vmin=0, vmax=max(20, np.nanmax(ckm_err) if not np.all(np.isnan(ckm_err)) else 20)) # Adjust vmax based on error
        axes[2].set_title('Error Percentage')
        for i in range(3):
            for j in range(3): axes[2].text(j, i, f"{ckm_err[i, j]:.2f}%", ha="center", va="center", color="w" if ckm_err[i, j] > 10 else "k")

        for ax in axes:
            ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
            ax.set_xticklabels(['d', 's', 'b']); ax.set_yticklabels(['u', 'c', 't'])
            ax.set_xlabel('Down-type Quarks'); ax.set_ylabel('Up-type Quarks')

        fig.colorbar(im1, ax=axes[0], label='Magnitude'); fig.colorbar(im2, ax=axes[1], label='Magnitude'); fig.colorbar(im3, ax=axes[2], label='Error (%)')
        plt.tight_layout()
        plot_path = "enhanced_cubic_ckm_matrix_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        return plot_path


    def create_comprehensive_visualization(self):
        """Create a comprehensive visualization (updated for cubic)."""
        if not self.is_optimized: print("Warning: Run optimization first."); return None
        if not self.results.get('success', False): print("Warning: Optimization failed.")


        fig = plt.figure(figsize=(15, 12))
        fig.suptitle("Comprehensive Model Results (Cubic Up-Quark Fit)", fontsize=16)

        # --- Plot 1: Running masses ---
        ax1 = fig.add_subplot(2, 2, 1)
        if self.results.get('success', False):
             mu_values_plot = np.logspace(0, 3, 100)
             running_masses = self.calculate_running_masses(mu_values_plot)
             for quark in self.quark_info:
                 ax1.loglog(running_masses['mu'], running_masses[quark], label=f'{quark}')
             # Add reference points (Predicted masses at ref scale)
             for quark in self.quark_info:
                 ref_scale = self.quark_info[quark]['ref_scale']
                 pred_mass = self.results.get('masses', {}).get(quark, np.nan)
                 ax1.scatter([ref_scale], [pred_mass], marker='o', s=30, label=f'_{quark} pred') # Underscore hides from main legend
             ax1.set_title('Running Quark Masses (Predicted)')
             ax1.set_xlabel('Energy Scale μ (GeV)'); ax1.set_ylabel('Running Mass (GeV)')
             ax1.grid(True, which='both', linestyle='--', alpha=0.7)
             ax1.legend(fontsize='small')
        else:
            ax1.text(0.5, 0.5, "Running masses not plotted\n(Optimization failed)", ha='center', va='center')
            ax1.set_title('Running Quark Masses (Predicted)')


        # --- Plot 2: CKM matrix ---
        ax2 = fig.add_subplot(2, 2, 2)
        ckm_pred = self.results.get('ckm', np.full((3,3), np.nan))
        im = ax2.imshow(ckm_pred, cmap='viridis', vmin=0, vmax=1)
        ax2.set_title('Predicted CKM Matrix')
        for i in range(3):
            for j in range(3): ax2.text(j, i, f"{ckm_pred[i, j]:.4f}", ha="center", va="center", color="w" if ckm_pred[i, j] < 0.5 else "k")
        ax2.set_xticks([0, 1, 2]); ax2.set_yticks([0, 1, 2])
        ax2.set_xticklabels(['d', 's', 'b']); ax2.set_yticklabels(['u', 'c', 't'])
        ax2.set_xlabel('Down-type'); ax2.set_ylabel('Up-type')
        fig.colorbar(im, ax=ax2, label='Magnitude', shrink=0.8)


        # --- Plot 3: Polynomial functions ---
        ax3 = fig.add_subplot(2, 2, 3)
        # Reuse logic from plot_polynomial_functions (log scale part)
        L_values = np.linspace(0, self.results.get('L_t', 5.0) * 1.1, 100)
        coeffs_up = self.results.get('c_coeffs_up_cubic', [0,0,0,0])
        up_masses = np.maximum(coeffs_up[0] * L_values**3 + coeffs_up[1] * L_values**2 + coeffs_up[2] * L_values + coeffs_up[3], 1e-9)
        coeffs_down = self.results.get('c_coeffs_down', [0,0,0])
        down_masses = np.maximum(coeffs_down[0] * L_values**2 + coeffs_down[1] * L_values + coeffs_down[2], 1e-9)
        gen_scales = self.results.get('gen_scale', [1,1,1])
        Ls = {q: self.results.get(f'L_{q}', 0) for q in self.quark_info}
        pred_masses = self.results.get('masses', {})

        ax3.semilogy(L_values, up_masses * gen_scales[2], 'r-', alpha=0.8, label='Up Poly*g3 (Cubic)')
        ax3.semilogy(L_values, down_masses * gen_scales[2], 'b-', alpha=0.8, label='Down Poly*g3 (Quad)')
        # Optionally add other generations with different linestyles/alpha
        ax3.scatter([Ls.get('u',np.nan)], [pred_masses.get('u',np.nan)], color='red', marker='o', s=50, label='u')
        ax3.scatter([Ls.get('c',np.nan)], [pred_masses.get('c',np.nan)], color='red', marker='s', s=50, label='c')
        ax3.scatter([Ls.get('t',np.nan)], [pred_masses.get('t',np.nan)], color='red', marker='^', s=50, label='t')
        ax3.scatter([Ls.get('d',np.nan)], [pred_masses.get('d',np.nan)], color='blue', marker='o', s=50, label='d')
        ax3.scatter([Ls.get('s',np.nan)], [pred_masses.get('s',np.nan)], color='blue', marker='s', s=50, label='s')
        ax3.scatter([Ls.get('b',np.nan)], [pred_masses.get('b',np.nan)], color='blue', marker='^', s=50, label='b')

        ax3.set_xlabel('Geodesic Length L')
        ax3.set_ylabel('Predicted Mass (GeV) - Log Scale')
        ax3.set_title('Polynomial Functions (Scaled)')
        ax3.grid(True, which='both', linestyle='--', alpha=0.7)
        ax3.legend(fontsize='small', ncol=2)
        min_m = min(m for m in pred_masses.values() if m is not None and m > 1e-9)
        max_m = max(m for m in pred_masses.values() if m is not None)
        if min_m is not None and max_m is not None: ax3.set_ylim(min_m*0.1, max_m*10)


        # --- Plot 4: Mass hierarchy ---
        ax4 = fig.add_subplot(2, 2, 4)
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        masses = [self.results.get('masses', {}).get(q, np.nan) for q in quarks]
        valid_indices = [i for i, m in enumerate(masses) if not np.isnan(m)]
        quarks_valid = [quarks[i] for i in valid_indices]
        masses_valid = [masses[i] for i in valid_indices]
        colors = ['r', 'b', 'g', 'c', 'm', 'y']
        colors_valid = [colors[i] for i in valid_indices]

        if masses_valid:
            ax4.bar(quarks_valid, masses_valid, color=colors_valid)
            ax4.set_yscale('log')
            ax4.set_title('Quark Mass Hierarchy (Predicted)')
            ax4.set_xlabel('Quark'); ax4.set_ylabel('Mass (GeV) - Log Scale')
            ax4.grid(True, axis='y', which='both', linestyle='--', alpha=0.7)
        else:
            ax4.text(0.5, 0.5, "Mass hierarchy not plotted\n(No valid mass data)", ha='center', va='center')
            ax4.set_title('Quark Mass Hierarchy (Predicted)')

        plt.tight_layout(rect=[0, 0.03, 1, 0.97]) # Adjust layout to prevent title overlap
        plot_path = "enhanced_cubic_comprehensive_visualization.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        return plot_path


# --- Main execution block ---
if __name__ == "__main__":
    # Create model instance
    model = EnhancedPolynomialModelCubicUp()

    # Optimize parameters
    print("Starting optimization...")
    results = model.optimize_parameters()
    print(f"Optimization finished. Success: {results.get('success')}")
    print(f"Message: {results.get('message')}")
    print(f"Final Objective Value: {results.get('final_objective_value', 'N/A')}")


    # Generate report and plots ONLY if optimization was successful
    # or if you want to see results with initial guesses on failure
    if model.is_optimized: # is_optimized is set to True only on success in this version
        print("\nGenerating report and plots...")
        try:
             report_path = model.generate_report()
             print(f"Report generated: {report_path}")
        except Exception as e:
            print(f"Error generating report: {e}")

        try:
             running_masses_plot, light_quarks_plot = model.plot_running_masses() # This calls optimize again if needed internally, maybe refactor
             print(f"Running masses plot: {running_masses_plot}")
             # print(f"Light quarks plot: {light_quarks_plot}") # Original script returned two paths here
        except Exception as e:
            print(f"Error generating running mass plots: {e}")

        try:
             alpha_s_plot = model.plot_alpha_s()
             print(f"Alpha_s plot: {alpha_s_plot}")
        except Exception as e:
            print(f"Error generating alpha_s plot: {e}")

        try:
             polynomial_functions_plot, polynomial_log_plot = model.plot_polynomial_functions()
             print(f"Polynomial functions plot: {polynomial_functions_plot}")
             print(f"Polynomial functions log plot: {polynomial_log_plot}")
        except Exception as e:
            print(f"Error generating polynomial plots: {e}")

        try:
             geodesic_lengths_plot = model.plot_geodesic_lengths()
             print(f"Geodesic lengths plot: {geodesic_lengths_plot}")
        except Exception as e:
            print(f"Error generating geodesic length plot: {e}")

        try:
             mass_hierarchy_plot = model.plot_mass_hierarchy()
             print(f"Mass hierarchy plot: {mass_hierarchy_plot}")
        except Exception as e:
            print(f"Error generating mass hierarchy plot: {e}")

        try:
             ckm_matrix_plot = model.plot_ckm_matrix()
             print(f"CKM matrix plot: {ckm_matrix_plot}")
        except Exception as e:
            print(f"Error generating CKM plot: {e}")

        try:
             comprehensive_plot = model.create_comprehensive_visualization()
             print(f"Comprehensive visualization: {comprehensive_plot}")
        except Exception as e:
            print(f"Error generating comprehensive plot: {e}")

    else:
        print("\nSkipping report and plot generation due to optimization failure.")
        # Optionally print results with initial parameters or failure state
        print("\nResults dictionary (contains initial parameters or last state before failure):")
        for key, value in results.items():
             if isinstance(value, np.ndarray):
                 print(f"  {key}: array shape {value.shape}")
             else:
                 print(f"  {key}: {value}")


    # --- Print summary (optional, report contains this info) ---
    if model.is_optimized:
        print("\n--- Summary from Optimized Results ---")
        # ... (Can add selected print statements from original if desired) ...
        print("\nMass Predictions (% Error):")
        for quark in model.quark_info:
            error = results.get('errors',{}).get(quark, np.nan)
            print(f"  {quark}: {error:.4f}%")

        print("\nCKM Errors (%):")
        ckm_err = results.get('ckm_errors', np.full((3,3), np.nan))
        for i in range(3): print(f"  {ckm_err[i,:]}")



SyntaxError: unterminated string literal (detected at line 518) (<ipython-input-1-30e20bbb6413>, line 518)